In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import glob
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder

BASE        = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUP_DIR  = BASE
ESC_DIR     = f'{BASE}/ESC-50-master/audio'
TEST_CSV    = f'{BASE}/test.csv'
SAMPLE_SUB  = f'{BASE}/sample_submission.csv'
EFF_CACHE   = '/kaggle/working/eff_cache'
AST_CACHE   = '/kaggle/working/ast_cache'
os.makedirs(EFF_CACHE, exist_ok=True)
os.makedirs(AST_CACHE, exist_ok=True)

GENRES   = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
SR       = 22050
SR_AST   = 16000
DURATION = 30
SEG_LEN  = 5
SEG_HOP  = 2
N_MELS   = 128
HOP      = 512
N_MFCC   = 40

le = LabelEncoder()
le.fit(GENRES)

test_df      = pd.read_csv(TEST_CSV)
esc_files    = glob.glob(f'{ESC_DIR}/*.wav')
filename_col = test_df.columns[1]

print(test_df.columns.tolist())
print(test_df.head(3))
print(f'ESC noise files: {len(esc_files)}')
print(f'Sample mashup path: {os.path.join(MASHUP_DIR, test_df.iloc[0][filename_col])}')

In [ ]:
# structure: genres_stems/<genre>/<song_id>/{drums,vocals,bass,others}.wav
STEMS = ['drums', 'vocals', 'bass', 'others']
records = []
for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    for song_id in os.listdir(genre_dir):
        song_dir = os.path.join(genre_dir, song_id)
        if not os.path.isdir(song_dir):
            continue
        for stem in STEMS:
            fp = os.path.join(song_dir, f'{stem}.wav')
            if os.path.exists(fp):
                records.append({'filepath': fp, 'genre': genre, 'song_id': song_id, 'stem': stem})

train_df          = pd.DataFrame(records)
train_df['label'] = le.transform(train_df['genre'])
print(train_df.shape)
print(train_df[['genre','stem']].value_counts().sort_index())

In [ ]:
# songs per genre and stems per song
songs_per_genre = train_df.groupby('genre')['song_id'].nunique()
plt.figure(figsize=(10, 4))
sns.barplot(x=songs_per_genre.index, y=songs_per_genre.values, palette='viridis')
plt.title('Songs per Genre (genres_stems)')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Song Count')
plt.tight_layout()
plt.show()
print(songs_per_genre)

In [ ]:
# audio duration across stems and genres
durations = []
for fp in train_df['filepath']:
    try:
        durations.append(librosa.get_duration(path=fp))
    except:
        durations.append(np.nan)
train_df['duration'] = durations

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train_df['duration'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Stem Duration Distribution')
axes[0].set_xlabel('Seconds')
train_df.boxplot(column='duration', by='stem', ax=axes[1], rot=0)
axes[1].set_title('Duration by Stem Type')
plt.suptitle('')
plt.tight_layout()
plt.show()
print(train_df.groupby('stem')['duration'].describe())

In [ ]:
# silence ratio check — stems like bass/others may have lots of silence
def silence_ratio(fp, top_db=30):
    try:
        y, sr = librosa.load(fp, sr=SR, mono=True)
        voiced = librosa.effects.split(y, top_db=top_db)
        voiced_samples = sum(e - s for s, e in voiced)
        return 1 - (voiced_samples / len(y))
    except:
        return np.nan

sample       = train_df.groupby('stem', group_keys=False).apply(lambda g: g.sample(min(50, len(g)), random_state=42))
sample['silence'] = sample['filepath'].apply(silence_ratio)

plt.figure(figsize=(9, 4))
sns.boxplot(data=sample, x='stem', y='silence', palette='Set2')
plt.title('Silence Ratio by Stem Type')
plt.ylabel('Silent Fraction')
plt.tight_layout()
plt.show()
print(sample.groupby('stem')['silence'].mean())

In [ ]:
# ESC-50 noise category breakdown
esc_meta = pd.read_csv(f'{BASE}/ESC-50-master/meta/esc50.csv')
print(f'ESC-50 total clips: {len(esc_meta)}')

plt.figure(figsize=(12, 4))
vc_esc = esc_meta['category'].value_counts().head(20)
sns.barplot(x=vc_esc.index, y=vc_esc.values, palette='magma')
plt.title('Top 20 ESC-50 Noise Categories')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# waveform + mel spectrogram — vocals stem, one per genre
fig, axes = plt.subplots(len(GENRES), 2, figsize=(14, len(GENRES) * 2.5))

for i, genre in enumerate(GENRES):
    row   = train_df[(train_df['genre'] == genre) & (train_df['stem'] == 'vocals')].iloc[0]
    y, sr = librosa.load(row['filepath'], sr=SR, duration=10)
    mel   = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64), ref=np.max)

    axes[i, 0].plot(y, linewidth=0.4, color='steelblue')
    axes[i, 0].set_title(f'{genre} vocals — waveform')
    axes[i, 0].axis('off')

    librosa.display.specshow(mel, sr=sr, ax=axes[i, 1], cmap='magma')
    axes[i, 1].set_title(f'{genre} vocals — mel spectrogram')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# most-frequent-class random baseline
sub_df          = pd.read_csv(SAMPLE_SUB)
most_common     = train_df['genre'].value_counts().idxmax()
sub_df['genre'] = most_common
sub_df.to_csv('/kaggle/working/submission_baseline.csv', index=False)
print(f'Baseline class: {most_common}')
print(sub_df.head())